In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

from Imputation_Methods import (
    mean_imputation,
    median_imputation,
    forward_fill_imputation,
    new_adaptive_weighted_seasonal_imputation,
    linear_interpolation
)

In [2]:
# Mengimport dataset
data2 = pd.read_csv(r"data/traffic.csv", sep=",")

In [3]:
# Fungsi untuk membuat missing value secara acak pada Series
def create_missing_data(series, missing_rate=0.15, random_state=42):
    """
    Membuat missing value secara acak pada Series.

    Returns:
        original_series : data asli
        missing_series  : data dengan missing value
        missing_indices : posisi nilai yang dibuat missing
    """

    original_series = series.copy()

    missing_series = series.copy()

    # Hanya memilih index yang awalnya tidak missing
    valid_indices = missing_series.dropna().index

    n_missing = int(len(valid_indices) * missing_rate)

    rng = np.random.default_rng(random_state)

    missing_indices = rng.choice(
        valid_indices,
        size=n_missing,
        replace=False
    )

    missing_series.loc[missing_indices] = np.nan

    return original_series, missing_series, missing_indices

In [4]:
original2, test2, missing2 = create_missing_data(
    data2["Vehicles"],
    missing_rate=0.15,
    random_state=42
)

In [5]:
print("\nDataset 2")
print("Data :", len(original2))
print("Missing :", test2.isna().sum())
print("Persentase :", test2.isna().mean() * 100)


Dataset 2
Data : 48120
Missing : 7218
Persentase : 15.0


In [6]:
# Import the evaluation function
from src.evaluation import evaluate_imputation

In [7]:
methods = {
    "Mean": {
        "function": mean_imputation,
        "params": {}
    },

    "Median": {
        "function": median_imputation,
        "params": {}
    },

    "Forward Fill": {
        "function": forward_fill_imputation,
        "params": {}
    },

    "New Adaptive Weighted Seasonal": {
        "function": new_adaptive_weighted_seasonal_imputation,
        "params": {
            "seasonal_period": 24
        }
    },
    
    "Linear Interpolation": {
        "function": linear_interpolation,
        "params": {}
    }
}

In [8]:
import time
import pandas as pd

results = []

for method_name, config in methods.items():

    # Mulai menghitung waktu
    start_time = time.perf_counter()

    result = evaluate_imputation(
        original2,
        test2,
        missing2,
        config["function"],
        **config["params"]
    )

    # Selesai menghitung waktu
    end_time = time.perf_counter()

    # Lama komputasi
    computation_time = end_time - start_time
    computation_minutes = computation_time / 60

    results.append({
        "Method": method_name,
        "MAE": result["MAE"],
        "RMSE": result["RMSE"],
        "Time (seconds)": round(computation_time, 4),
        "Time (minutes)": round(computation_minutes, 4)
    })

# Buat tabel hasil
results_df = pd.DataFrame(results)

results_df

,Method,MAE,RMSE,Time (seconds),Time (minutes)
0,Mean,15.401598,20.647394,0.0125,0.0002
1,Median,13.970768,22.095805,0.0035,0.0001
2,Forward Fill,3.456775,5.217771,0.0060,0.0001
3,New Adaptive Weighted Seasonal,3.256817,5.075809,0.4526,0.0075
4,Linear Interpolation,2.404766,3.543006,0.0071,0.0001


In [9]:
results_df = results_df.sort_values(
    by="MAE",
    ascending=True
).reset_index(drop=True)

results_df

,Method,MAE,RMSE,Time (seconds),Time (minutes)
0,Linear Interpolation,2.404766,3.543006,0.0071,0.0001
1,New Adaptive Weighted Seasonal,3.256817,5.075809,0.4526,0.0075
2,Forward Fill,3.456775,5.217771,0.0060,0.0001
3,Median,13.970768,22.095805,0.0035,0.0001
4,Mean,15.401598,20.647394,0.0125,0.0002
